## 1. Import Libraries & Load Data

In [ ]:
# Import required libraries
import os
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate, StratifiedKFold, train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import make_scorer
import time
import warnings
warnings.filterwarnings('ignore')


# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

In [ ]:
# Load the dataset
current_user = os.getlogin()
PROCESSED_DIR = Path(f"/home/{current_user}/local-share/processed")
df = pd.read_csv(PROCESSED_DIR / 'feature_selection.csv')

print("Dataset shape:", df.shape)
print("\nDataset info:")
print(df.info())

## 2. Prepare Data

**Purpose**: Split data into train/validation/test, define target feature and prepare all features.

In [ ]:
# Split data based on the 'set' column
train_df = df[df['set'] == 'train'].copy()
val_df = df[df['set'] == 'val'].copy()
test_df = df[df['set'] == 'test'].copy()

print(f"Training set shape: {train_df.shape}")
print(f"Validation set shape: {val_df.shape}")
print(f"Test set shape: {test_df.shape}")

In [ ]:
# Specify the target variable
target_col = 'sdo5_degree_drop_out'

# Exclude non-feature columns (set, target, any IDs)
exclude_cols = ['set', target_col]

# Get feature columns
feature_cols = [col for col in df.columns if col not in exclude_cols]
print(f"Number of features: {len(feature_cols)}")
print(f"Feature columns: {feature_cols}")

# Prepare training data
X_train = train_df[feature_cols]
y_train = train_df[target_col]

# Prepare validation data
X_val = val_df[feature_cols]
y_val = val_df[target_col]

# Prepare test data
X_test = test_df[feature_cols]
y_test = test_df[target_col]

## 3. Feature Scaling for Neural Networks

**Purpose**: Standardize features to improve neural network performance and convergence.

**Method**:
- StandardScaler: zero mean, unit variance transformation
- Fit scaler on training data only
- Transform train/validation/test sets consistently
- Prevents data leakage by using training statistics only
- Critical for neural networks due to gradient-based optimization

In [ ]:
# Neural Networks are very sensitive to feature scaling
# Scale features using StandardScaler (fit on training data only)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed!")
print(f"Training features shape after scaling: {X_train_scaled.shape}")
print(f"Validation features shape after scaling: {X_val_scaled.shape}")
print(f"Test features shape after scaling: {X_test_scaled.shape}")

# Convert back to DataFrames for easier handling (optional)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_cols, index=X_train.index)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=feature_cols, index=X_val.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_cols, index=X_test.index)

print(f"\nFeature scaling statistics (training set):")
print(f"Mean: {X_train_scaled.mean().mean():.6f}")
print(f"Std: {X_train_scaled.std().mean():.6f}")

In [ ]:
# Check for missing values in scaled data
print(f"Missing values in X_train_scaled: {X_train_scaled.isna().sum().sum()}")
print(f"Missing values in X_val_scaled: {X_val_scaled.isna().sum().sum()}")
print(f"Missing values in X_test_scaled: {X_test_scaled.isna().sum().sum()}")

# If there are missing values, impute them with median
if X_train_scaled.isna().any().any():
    print("\nImputing missing values with median strategy...")
    imputer = SimpleImputer(strategy='median')
    X_train_scaled_imputed = pd.DataFrame(
        imputer.fit_transform(X_train_scaled),
        columns=feature_cols,
        index=X_train_scaled.index
    )
    X_val_scaled_imputed = pd.DataFrame(
        imputer.transform(X_val_scaled),
        columns=feature_cols,
        index=X_val_scaled.index
    )
    X_test_scaled_imputed = pd.DataFrame(
        imputer.transform(X_test_scaled),
        columns=feature_cols,
        index=X_test_scaled.index
    )
    print("Missing values imputed successfully!")
else:
    X_train_scaled_imputed = X_train_scaled
    X_val_scaled_imputed = X_val_scaled
    X_test_scaled_imputed = X_test_scaled
    print("No missing values found - proceeding with original data")

# Update the scaled datasets to use imputed versions
X_train_scaled = X_train_scaled_imputed
X_val_scaled = X_val_scaled_imputed
X_test_scaled = X_test_scaled_imputed

## 4. Stratified 10-Fold Cross-Validation Training and Evaluation

**Purpose**: Train and evaluate neural network using stratified 10-fold cross-validation.

**Methodology**:
- Combine train+validation sets for CV (test set remains separate)
- Stratified K-fold maintains class distribution across folds
- Multiple metrics: accuracy, ROC AUC, precision, recall, F1-score
- Class balancing with balanced class weights approach
- Default hyperparameters for baseline performance assessment
- Early stopping to prevent overfitting

In [ ]:
# Combine training and validation data for cross-validation
# (Keep test set separate for final evaluation)
X_train_val = pd.concat([X_train_scaled, X_val_scaled], axis=0)
y_train_val = pd.concat([y_train, y_val], axis=0)

print(f"\nCombined training + validation set shape: {X_train_val.shape}")
print(f"Target distribution in combined set:")
print(y_train_val.value_counts(normalize=True))

# Set up stratified 10-fold cross-validation
cv_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Define scoring metrics for cross-validation
scoring_metrics = {
    'accuracy': 'accuracy',
    'roc_auc': 'roc_auc',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1'
}

# Calculate class weights for balanced learning
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(y_train_val),
    y=y_train_val
)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

print(f"Class weights: {class_weight_dict}")

# Initialize the neural network model
# Using a simple architecture: one hidden layer with 100 neurons
cv_model = MLPClassifier(
    hidden_layer_sizes=(100,),  # One hidden layer with 100 neurons
    activation='relu',          # ReLU activation function
    solver='adam',             # Adam optimizer (good default)
    alpha=0.0001,              # L2 regularization
    learning_rate='constant',   # Fixed learning rate
    learning_rate_init=0.001,  # Initial learning rate
    max_iter=500,              # Maximum iterations
    early_stopping=True,       # Stop early if no improvement
    validation_fraction=0.1,   # 10% for early stopping validation
    n_iter_no_change=20,       # Stop if no improvement for 20 iterations
    random_state=42
)

print("\nStarting stratified 10-fold cross-validation...")
print("This will train and evaluate 10 different neural networks...")
print(f"Network architecture: {cv_model.hidden_layer_sizes}")

start_time = time.time()

# Perform cross-validation with custom class weighting
# We'll implement custom scoring that handles class weights
cv_results = cross_validate(
    cv_model,
    X_train_val,
    y_train_val,
    cv=cv_strategy,
    scoring=scoring_metrics,
    return_train_score=True,
    n_jobs=1  # Neural networks don't parallelize well across jobs
)

end_time = time.time()
print(f"Cross-validation completed in {end_time - start_time:.2f} seconds")

## 5. Cross-Validation Results Analysis

**Purpose**: Analyze and interpret 10-fold cross-validation performance metrics.

**Analysis**:
- Statistical summary: mean, std, min, max, median for each metric
- Performance consistency across folds
- Key metrics extraction for downstream analysis
- Confidence assessment through variance analysis

In [ ]:
# Analyze and visualize cross-validation results
from scipy import stats

print("=" * 80)
print("10-FOLD CROSS-VALIDATION RESULTS")
print("=" * 80)

# Extract test scores (what we care about for model evaluation)
test_scores = {
    'Accuracy': cv_results['test_accuracy'],
    'ROC AUC': cv_results['test_roc_auc'],
    'Precision': cv_results['test_precision'],
    'Recall': cv_results['test_recall'],
    'F1-Score': cv_results['test_f1']
}

# Calculate statistics for each metric
cv_stats = {}
for metric, scores in test_scores.items():
    cv_stats[metric] = {
        'mean': np.mean(scores),
        'std': np.std(scores),
        'min': np.min(scores),
        'max': np.max(scores),
        'median': np.median(scores)
    }

# Display summary statistics
print(f"\n📊 CROSS-VALIDATION PERFORMANCE SUMMARY:")
print(f"{'Metric':<12} {'Mean':<8} {'Std':<8} {'Min':<8} {'Max':<8} {'Median':<8}")
print("-" * 60)
for metric, stats in cv_stats.items():
    print(f"{metric:<12} {stats['mean']:.3f}   {stats['std']:.3f}   {stats['min']:.3f}   {stats['max']:.3f}   {stats['median']:.3f}")

# Store main CV results for later use
cv_mean_accuracy = cv_stats['Accuracy']['mean']
cv_mean_roc_auc = cv_stats['ROC AUC']['mean']

print(f"\n🎯 KEY METRICS:")
print(f"   • Mean CV Accuracy: {cv_mean_accuracy:.3f} ± {cv_stats['Accuracy']['std']:.3f}")
print(f"   • Mean CV ROC AUC: {cv_mean_roc_auc:.3f} ± {cv_stats['ROC AUC']['std']:.3f}")

print("=" * 80)

## 6. Train Final Model and Validation Analysis

**Purpose**: Train final model on full train+validation set and perform holdout validation.

**Process**:
- Train final model on combined train+validation data
- Create holdout subset for visualization and comparison
- Generate confusion matrix and classification metrics
- Compare holdout results with CV performance for consistency
- Analyze training convergence and potential overfitting

In [ ]:
# Train final model on full training+validation set for analysis and final evaluation
print("Training final neural network model on combined training+validation data...")

nn_model = MLPClassifier(
    hidden_layer_sizes=(100,),  # One hidden layer with 100 neurons
    activation='relu',          # ReLU activation function
    solver='adam',             # Adam optimizer
    alpha=0.0001,              # L2 regularization
    learning_rate='constant',   # Fixed learning rate
    learning_rate_init=0.001,  # Initial learning rate
    max_iter=500,              # Maximum iterations
    early_stopping=True,       # Stop early if no improvement
    validation_fraction=0.1,   # 10% for early stopping validation
    n_iter_no_change=20,       # Stop if no improvement for 20 iterations
    random_state=42
)

# Fit on the combined train+val set
nn_model.fit(X_train_val, y_train_val)

print(f"Final model trained successfully!")
print(f"Number of features used: {nn_model.n_features_in_}")
print(f"Number of iterations: {nn_model.n_iter_}")
print(f"Number of layers: {nn_model.n_layers_}")
print(f"Hidden layer sizes: {nn_model.hidden_layer_sizes}")
print(f"Final loss: {nn_model.loss_:.6f}")

# For comparison with CV results, let's create a validation subset prediction
# Using a simple holdout for visualization purposes
X_temp, X_val_holdout, y_temp, y_val_holdout = train_test_split(
    X_train_val, y_train_val, test_size=0.2, random_state=42, stratify=y_train_val
)

# Train on temp set, predict on holdout
nn_temp = MLPClassifier(
    hidden_layer_sizes=(100,),
    activation='relu',
    solver='adam',
    alpha=0.0001,
    learning_rate='constant',
    learning_rate_init=0.001,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=42
)
nn_temp.fit(X_temp, y_temp)

y_val_pred = nn_temp.predict(X_val_holdout)
y_val_pred_proba = nn_temp.predict_proba(X_val_holdout)[:, 1]

# Calculate validation metrics for comparison
val_accuracy = accuracy_score(y_val_holdout, y_val_pred)
val_roc_auc = roc_auc_score(y_val_holdout, y_val_pred_proba)

print(f"\n=== Holdout Validation Performance (for comparison) ===")
print(f"Holdout Accuracy: {val_accuracy:.4f} (CV Mean: {cv_mean_accuracy:.4f})")
print(f"Holdout ROC AUC: {val_roc_auc:.4f} (CV Mean: {cv_mean_roc_auc:.4f})")
print(f"Holdout model iterations: {nn_temp.n_iter_}")
print(f"Holdout model final loss: {nn_temp.loss_:.6f}")

print(f"\n=== Classification Report (Holdout) ===")
print(classification_report(y_val_holdout, y_val_pred))

# Confusion Matrix for holdout
print(f"\n=== Confusion Matrix (Holdout) ===")
cm = confusion_matrix(y_val_holdout, y_val_pred)
print(cm)

# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Confusion Matrix - Holdout Validation Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

## 7. ROC Curve Analysis

**Purpose**: Visualize model's discriminative performance using ROC curve.

**Analysis**:
- Plot ROC curve for holdout validation set
- Compare with random baseline (diagonal line)
- Display AUC score for quantitative assessment
- Compare holdout AUC with cross-validation mean for consistency

In [ ]:
# Plot ROC Curve for holdout validation
fpr, tpr, thresholds = roc_curve(y_val_holdout, y_val_pred_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {val_roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random baseline')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Neural Network Model (Holdout Validation)')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

# Add note about CV vs holdout
print(f"Note: This ROC curve is based on a single holdout validation.")
print(f"The cross-validation results provide a more robust estimate:")
print(f"CV Mean ROC AUC: {cv_mean_roc_auc:.4f} ± {cv_stats['ROC AUC']['std']:.4f}")

## 8. Neural Network Architecture Analysis

**Purpose**: Examine neural network architecture, weights, and training characteristics.

**Analysis**:
- Display network architecture details
- Analyze training convergence and loss curve
- Examine weight statistics across layers
- Discuss network complexity and potential overfitting indicators

In [ ]:
# Analyze neural network architecture and training characteristics
print("=" * 80)
print("NEURAL NETWORK ARCHITECTURE ANALYSIS")
print("=" * 80)

print(f"\n🧠 NETWORK ARCHITECTURE:")
print(f"   • Input layer: {nn_model.n_features_in_} neurons (features)")
print(f"   • Hidden layers: {nn_model.hidden_layer_sizes}")
print(f"   • Output layer: {nn_model.n_outputs_} neuron (binary classification)")
print(f"   • Total layers: {nn_model.n_layers_}")
print(f"   • Activation function: {nn_model.activation}")
print(f"   • Solver: {nn_model.solver}")

# Calculate total number of parameters
total_params = 0
layer_sizes = [nn_model.n_features_in_] + list(nn_model.hidden_layer_sizes) + [1]
for i in range(len(layer_sizes) - 1):
    # Weights + biases
    weights = layer_sizes[i] * layer_sizes[i + 1]
    biases = layer_sizes[i + 1]
    layer_params = weights + biases
    total_params += layer_params
    print(f"   • Layer {i+1}: {layer_sizes[i]} → {layer_sizes[i+1]} = {layer_params:,} parameters")

print(f"   • Total parameters: {total_params:,}")

print(f"\n📈 TRAINING CHARACTERISTICS:")
print(f"   • Training iterations: {nn_model.n_iter_}")
print(f"   • Final loss: {nn_model.loss_:.6f}")
print(f"   • Learning rate: {nn_model.learning_rate_init}")
print(f"   • L2 regularization (alpha): {nn_model.alpha}")
print(f"   • Early stopping: {nn_model.early_stopping}")

# Analyze weight statistics if available
if hasattr(nn_model, 'coefs_'):
    print(f"\n⚖️  WEIGHT STATISTICS:")
    for i, coef in enumerate(nn_model.coefs_):
        weight_mean = np.mean(np.abs(coef))
        weight_std = np.std(coef)
        weight_max = np.max(np.abs(coef))
        print(f"   • Layer {i+1} weights - Mean: {weight_mean:.4f}, Std: {weight_std:.4f}, Max: {weight_max:.4f}")

# Learning curve visualization (if loss curve is available)
# Note: MLPClassifier doesn't store loss history by default, but we can discuss convergence
print(f"\n🎯 CONVERGENCE ANALYSIS:")
if nn_model.n_iter_ < nn_model.max_iter:
    print(f"   • ✅ Model converged after {nn_model.n_iter_} iterations")
    print(f"   • Early stopping activated (no improvement for {nn_model.n_iter_no_change} iterations)")
else:
    print(f"   • ⚠️  Model reached maximum iterations ({nn_model.max_iter})")
    print(f"   • May benefit from more training time or different learning rate")

# Model complexity assessment
print(f"\n🔍 COMPLEXITY ASSESSMENT:")
data_points = len(X_train_val)
params_to_data_ratio = total_params / data_points
print(f"   • Data points: {data_points:,}")
print(f"   • Parameters to data ratio: {params_to_data_ratio:.4f}")
if params_to_data_ratio > 0.1:
    print(f"   • ⚠️  High parameter-to-data ratio - potential overfitting risk")
else:
    print(f"   • ✅ Reasonable parameter-to-data ratio")

print("=" * 80)

## 9. Final Evaluation on Test Set

**Purpose**: Evaluate final model performance on completely unseen test data.

In [ ]:
# Use the final neural network model for evaluation
final_model = nn_model
print("Using final neural network model for test evaluation")

# Make predictions on test set
y_test_pred = final_model.predict(X_test_scaled)
y_test_pred_proba = final_model.predict_proba(X_test_scaled)[:, 1]

# Calculate test metrics
test_accuracy = accuracy_score(y_test, y_test_pred)
test_roc_auc = roc_auc_score(y_test, y_test_pred_proba)

print("\n=== Final Test Set Performance ===")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test ROC AUC Score: {test_roc_auc:.4f}")

print("\n=== Test Set Classification Report ===")
print(classification_report(y_test, y_test_pred))

# Test set confusion matrix
print("\n=== Test Set Confusion Matrix ===")
test_cm = confusion_matrix(y_test, y_test_pred)
print(test_cm)

# Visualize test confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(test_cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Confusion Matrix - Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# Summary of model performance across all sets
print("\n=== Model Performance Summary ===")
print(f"Cross-Validation Accuracy: {cv_mean_accuracy:.4f} ± {cv_stats['Accuracy']['std']:.4f}")
print(f"Cross-Validation ROC AUC: {cv_mean_roc_auc:.4f} ± {cv_stats['ROC AUC']['std']:.4f}")
print(f"Holdout Validation Accuracy: {val_accuracy:.4f}")
print(f"Holdout Validation ROC AUC: {val_roc_auc:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test ROC AUC: {test_roc_auc:.4f}")

## 10. Comprehensive Model Evaluation and Analysis

**Purpose**: Provide complete performance analysis, business impact assessment, and recommendations.

In [ ]:
# Comprehensive Model Evaluation Analysis

print("=" * 80)
print("COMPREHENSIVE MODEL EVALUATION ANALYSIS")
print("=" * 80)

# Calculate baseline accuracy (predicting majority class)
test_dropout_rate = y_test.mean()
baseline_accuracy = max(test_dropout_rate, 1 - test_dropout_rate)

print(f"\n📊 DATASET CHARACTERISTICS:")
print(f"   • Test set size: {len(y_test):,} students")
print(f"   • Dropout rate: {test_dropout_rate:.1%}")
print(f"   • Non-dropout rate: {1-test_dropout_rate:.1%}")
print(f"   • Baseline accuracy (majority class): {baseline_accuracy:.1%}")

print(f"\n🎯 MODEL PERFORMANCE vs BASELINE:")
print(f"   • Test Accuracy: {test_accuracy:.1%} (vs {baseline_accuracy:.1%} baseline)")
print(f"   • Accuracy improvement: {test_accuracy - baseline_accuracy:+.1%}")
print(f"   • ROC AUC: {test_roc_auc:.3f} (0.5 = random, 1.0 = perfect)")

# Performance analysis
if test_accuracy > baseline_accuracy:
    print(f"   ✅ Model beats baseline by {test_accuracy - baseline_accuracy:.1%}")
else:
    print(f"   ❌ Model underperforms baseline by {baseline_accuracy - test_accuracy:.1%}")

# ROC AUC interpretation
if test_roc_auc > 0.8:
    auc_rating = "Excellent"
elif test_roc_auc > 0.7:
    auc_rating = "Good"
elif test_roc_auc > 0.6:
    auc_rating = "Fair"
else:
    auc_rating = "Poor"

print(f"   📈 AUC Rating: {auc_rating}")

# Cross-validation consistency analysis
cv_test_diff = abs(cv_mean_accuracy - test_accuracy)
if cv_test_diff < 0.02:
    consistency_rating = "Excellent"
elif cv_test_diff < 0.05:
    consistency_rating = "Good"
elif cv_test_diff < 0.1:
    consistency_rating = "Fair"
else:
    consistency_rating = "Poor"

print(f"   🎯 CV-Test Consistency: {consistency_rating} (diff: {cv_test_diff:.3f})")

# Confusion Matrix Analysis
tn, fp, fn, tp = test_cm.ravel()
sensitivity = tp / (tp + fn)  # True Positive Rate (Recall)
specificity = tn / (tn + fp)  # True Negative Rate
precision = tp / (tp + fp)    # Positive Predictive Value
npv = tn / (tn + fn)         # Negative Predictive Value

print(f"\n🔍 DETAILED PERFORMANCE METRICS:")
print(f"   • Sensitivity (Recall): {sensitivity:.1%} - Correctly identified dropouts")
print(f"   • Specificity: {specificity:.1%} - Correctly identified non-dropouts")
print(f"   • Precision: {precision:.1%} - Accuracy of dropout predictions")
print(f"   • Negative Predictive Value: {npv:.1%} - Accuracy of non-dropout predictions")

print(f"\n🎭 CONFUSION MATRIX BREAKDOWN:")
print(f"   • True Negatives: {tn:,} (Correctly predicted non-dropouts)")
print(f"   • False Positives: {fp:,} (Incorrectly predicted as dropouts)")
print(f"   • False Negatives: {fn:,} (Missed dropouts)")
print(f"   • True Positives: {tp:,} (Correctly predicted dropouts)")

# Business Impact Analysis
missed_dropouts_rate = fn / (tp + fn)
false_alarm_rate = fp / (tn + fp)

print(f"\n💼 BUSINESS IMPACT:")
print(f"   • Missed dropout rate: {missed_dropouts_rate:.1%} ({fn:,} students)")
print(f"   • False alarm rate: {false_alarm_rate:.1%} ({fp:,} students)")

print(f"\n🧠 NEURAL NETWORK CONFIGURATION:")
print(f"   • Architecture: {final_model.n_features_in_} → {final_model.hidden_layer_sizes[0]} → 1")
print(f"   • Total parameters: {total_params:,}")
print(f"   • Activation: {final_model.activation}")
print(f"   • Solver: {final_model.solver}")
print(f"   • Learning rate: {final_model.learning_rate_init}")
print(f"   • L2 regularization: {final_model.alpha}")
print(f"   • Training iterations: {final_model.n_iter_}")
print(f"   • Early stopping: {final_model.early_stopping}")

print(f"\n🎯 RECOMMENDATIONS:")
if test_accuracy < baseline_accuracy:
    print("   ❗ Model performance is below baseline - consider:")
    print("     - Different network architectures (more/fewer layers)")
    print("     - Hyperparameter tuning (learning rate, regularization)")
    print("     - Different activation functions (tanh, sigmoid)")
    print("     - Feature engineering and selection")
    print("     - Ensemble methods or different algorithms")
elif test_roc_auc < 0.7:
    print("   ⚠️  Model shows limited predictive power - consider:")
    print("     - Deeper networks (more hidden layers)")
    print("     - Different architectures (wider layers)")
    print("     - Hyperparameter optimization")
    print("     - Feature interaction terms or polynomial features")
    print("     - Regularization tuning (dropout, weight decay)")
else:
    print("   ✅ Model shows reasonable performance but could be improved:")
    print("     - Architecture optimization (GridSearchCV for layer sizes)")
    print("     - Advanced optimizers (RMSprop, different learning schedules)")
    print("     - Ensemble methods (bagging multiple neural networks)")
    print("     - Feature selection and engineering")
    print("     - Cross-validation for hyperparameter tuning")

if final_model.n_iter_ >= final_model.max_iter:
    print("   ⚠️  Training may be incomplete:")
    print("     - Increase max_iter for longer training")
    print("     - Adjust learning rate (lower for more stable convergence)")
    print("     - Monitor loss curve for convergence patterns")

print(f"\n🔗 NEURAL NETWORK INSIGHTS:")
print("   • Non-linear model - can capture complex feature interactions")
print("   • Requires careful tuning of architecture and hyperparameters")
print("   • Sensitive to feature scaling (already applied)")
print("   • Black-box nature limits interpretability compared to linear models")
print("   • This experiment uses simple default architecture for baseline exploration")
print("   • Consider deeper networks or different architectures for improved performance")

print("\n" + "=" * 80)